In [1]:
import socket
print(socket.gethostname())

hawk-gpu6


In [2]:
import sys
print(sys.executable)

/home/maska/.conda/envs/torch_env/bin/python


In [3]:
!nvidia-smi

Mon Mar 30 09:42:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.163.01             Driver Version: 550.163.01     CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-PCIE-40GB          Off |   00000000:3B:00.0 Off |                    0 |
| N/A   44C    P0             35W /  250W |       1MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [4]:
import torch
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
print("GPU name:", torch.cuda.get_device_name(0))

Torch version: 2.5.1+cu121
CUDA available: True
GPU count: 4
GPU name: NVIDIA A100-PCIE-40GB


In [5]:
from pathlib import Path

#BASE_DIR = Path.home() / "Aryan&Sunayana" / "data1"
BASE_DIR = Path.home() / "NEW_poledata" 
print("BASE_DIR:", BASE_DIR)
print("train/images exists:", (BASE_DIR / "train" / "images").exists())
print("train/labels exists:", (BASE_DIR / "train" / "labels").exists())
print("valid/images exists:", (BASE_DIR / "valid" / "images").exists())
print("test/images exists:", (BASE_DIR / "test" / "images").exists())

BASE_DIR: /home/maska/NEW_poledata
train/images exists: True
train/labels exists: True
valid/images exists: True
test/images exists: True


In [6]:

from IPython.display import Image
import ultralytics 
ultralytics.checks()

Ultralytics 8.4.19 🚀 Python-3.10.18 torch-2.5.1+cu121 CUDA:0 (NVIDIA A100-PCIE-40GB, 40326MiB)
Setup complete ✅ (64 CPUs, 376.6 GB RAM, 92.4/98.3 GB disk)


In [9]:
# ── Install dependencies (run this cell first in Colab or server) ──────────────
# !pip install ultralytics opencv-python pyyaml torch torchvision -q

import os
import yaml
import torch
from pathlib import Path
from typing import List, Optional
from ultralytics import YOLO

# ─────────────────────────────────────────────────────────────────────
# MULTI-GPU CONFIGURATION & UTILITIES
# ─────────────────────────────────────────────────────────────────────

class GPUManager:
    """
    Manages GPU detection, idle GPU identification, and multi-GPU allocation.
    Optimizes for server deployments with multiple GPUs.
    """

    def __init__(self):
        self.available_gpus = self._detect_gpus()
        self.idle_gpus      = self._identify_idle_gpus()
        self.selected_gpus  = None

    def _detect_gpus(self) -> List[int]:
        """Detect all available GPUs on the system."""
        if not torch.cuda.is_available():
            print("[WARN] CUDA not available. Training will run on CPU.")
            return []

        num_gpus = torch.cuda.device_count()
        gpus     = list(range(num_gpus))

        print(f"\n╔══════════════════════════════════════════════════════╗")
        print(f"║ GPU DETECTION REPORT                                 ║")
        print(f"╚══════════════════════════════════════════════════════╝")
        print(f"\n  Total GPUs Available: {num_gpus}")

        for gpu_id in gpus:
            gpu_name   = torch.cuda.get_device_name(gpu_id)
            gpu_memory = torch.cuda.get_device_properties(gpu_id).total_memory / 1e9
            print(f"\n  GPU {gpu_id}:")
            print(f"    ├─ Name  : {gpu_name}")
            print(f"    └─ Memory: {gpu_memory:.2f} GB")

        return gpus

    def _identify_idle_gpus(self) -> List[int]:
        """
        Identify idle GPUs by checking memory usage.
        A GPU is considered idle if memory usage < 10% of total.
        """
        if not self.available_gpus:
            return []

        idle_gpus = []
        print(f"\n╔══════════════════════════════════════════════════════╗")
        print(f"║ GPU IDLE STATUS CHECK                                ║")
        print(f"╚══════════════════════════════════════════════════════╝\n")

        for gpu_id in self.available_gpus:
            try:
                torch.cuda.reset_peak_memory_stats(gpu_id)
                current_memory = torch.cuda.memory_allocated(gpu_id) / 1e9
                total_memory   = torch.cuda.get_device_properties(gpu_id).total_memory / 1e9
                memory_percent = (current_memory / total_memory) * 100

                status = "IDLE ✓" if memory_percent < 10 else "IN USE ✗"
                print(f"  GPU {gpu_id}: {memory_percent:.1f}% used [{status}]")

                if memory_percent < 10:
                    idle_gpus.append(gpu_id)

            except Exception as e:
                print(f"  [ERROR] Could not check GPU {gpu_id}: {str(e)}")

        return idle_gpus

    def select_gpus(self, num_gpus: Optional[int] = None) -> List[int]:
        """
        Select GPUs for training.

        Args:
            num_gpus: Number of GPUs to use. If None, use all idle GPUs.

        Returns:
            List of GPU IDs to use for training.
        """
        if not self.available_gpus:
            print("\n[ERROR] No GPUs available. CPU training will be very slow.")
            self.selected_gpus = []
            return []

        if not self.idle_gpus:
            print(f"\n[WARN] No idle GPUs detected. Using first GPU anyway.")
            self.selected_gpus = [self.available_gpus[0]]
        else:
            self.selected_gpus = self.idle_gpus if num_gpus is None else self.idle_gpus[:num_gpus]

        print(f"\n╔══════════════════════════════════════════════════════╗")
        print(f"║ GPU SELECTION FOR TRAINING                           ║")
        print(f"╚══════════════════════════════════════════════════════╝")
        print(f"\n  Selected GPUs : {self.selected_gpus}")
        print(f"  Count         : {len(self.selected_gpus)}")

        return self.selected_gpus

    def get_device_string(self) -> str:
        """
        Return device string for YOLO training.

        Returns:
            Comma-separated GPU IDs (e.g. "0,1,2,3") or "cpu".
        """
        if not self.selected_gpus:
            return "cpu"
        return ",".join(map(str, self.selected_gpus))


# ─────────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────────

HOME_DIR = Path.home()
BASE_DIR        = HOME_DIR / "NEW_poledata"

TRAIN_IMG_DIR   = BASE_DIR / "train" / "images"
TRAIN_LBL_DIR   = BASE_DIR / "train" / "labels"
VALID_IMG_DIR   = BASE_DIR / "valid" / "images"
VALID_LBL_DIR   = BASE_DIR / "valid" / "labels"
TEST_IMG_DIR    = BASE_DIR / "test"  / "images"
TEST_LBL_DIR    = BASE_DIR / "test"  / "labels"



# data.yaml
DATA_YAML   = BASE_DIR / "data.yaml"



# YOLOv8 settings — OPTIMIZED
MODEL_WEIGHTS   = "yolov8l.pt"        
PROJECT_NAME    = str(HOME_DIR / "gsv_pole_detection1")
RUN_NAME              = "exp1_optimized_multiGPU"
EPOCHS                = 100
IMG_SIZE              = 640          # Better for thin vertical objects
BATCH_SIZE_PER_GPU    = 16           # Scales with GPU count; increase if memory allows
NUM_GPUS              = None         # None = auto-select all idle GPUs
WORKERS               = 8           # Increased for A100 multi-GPU setup


# ─────────────────────────────────────────────────────────────────────
# STEP 0 — Auto-fix data.yaml to use absolute paths
# ─────────────────────────────────────────────────────────────────────

def fix_data_yaml():
    """
    Rewrites data.yaml with absolute paths so YOLOv8 can find the data
    regardless of the working directory.
    """
    if not DATA_YAML.exists():
        raise FileNotFoundError(f"data.yaml not found at: {DATA_YAML}")

    with open(DATA_YAML, "r") as f:
        data = yaml.safe_load(f)

    data["train"] = str(TRAIN_IMG_DIR)
    data["val"]   = str(VALID_IMG_DIR)
    data["test"]  = str(TEST_IMG_DIR)

    with open(DATA_YAML, "w") as f:
        yaml.dump(data, f, default_flow_style=False)

    print(f"[Step 0] data.yaml fixed with absolute paths:")
    print(f"  train → {TRAIN_IMG_DIR}")
    print(f"  val   → {VALID_IMG_DIR}")
    print(f"  test  → {TEST_IMG_DIR}")


# ─────────────────────────────────────────────────────────────────────
# STEP 1 — YOLOv8 Training (GPU-native, on-the-fly augmentation)
# ─────────────────────────────────────────────────────────────────────

def train_yolov8(gpu_manager: GPUManager):
    """
    Train YOLOv8 directly on the original dataset.

    All augmentation is handled on-the-fly by YOLO's built-in pipeline —
    no CPU pre-processing, no disk I/O overhead, full GPU utilisation.

    KEY DECISIONS:
      • No Albumentations: eliminated CPU bottleneck & disk writes
      • device="0,1,2,3": DDP automatically engaged by Ultralytics
      • batch = BATCH_SIZE_PER_GPU × num_gpus: maximises throughput
      • imgsz=960: critical for detecting thin poles
      • box=9.0: forces tighter bounding boxes
      • cos_lr=True: smoother decay schedule
      • sync_bn=True: stable batch-norm across GPUs
      • amp=True: mixed-precision for memory efficiency
    """
    model = YOLO(MODEL_WEIGHTS)

    num_gpus     = max(len(gpu_manager.selected_gpus), 1) if gpu_manager.selected_gpus else 1
    total_batch  = BATCH_SIZE_PER_GPU * num_gpus
    device_str   = gpu_manager.get_device_string()

    print(f"\n[Step 1] Starting YOLOv8 Multi-GPU Training")
    print(f"  Model      : {MODEL_WEIGHTS}")
    print(f"  Epochs     : {EPOCHS}")
    print(f"  Image size : {IMG_SIZE}px")
    print(f"  GPUs       : {device_str}")
    print(f"  Batch size : {total_batch}  ({BATCH_SIZE_PER_GPU} × {num_gpus} GPUs)")
    print(f"  Workers    : {WORKERS}")
    print(f"  Data YAML  : {DATA_YAML}\n")

    results = model.train(
        data=str(DATA_YAML),        # ← original dataset, no augmented copies
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=total_batch,
        device=device_str,          # e.g. "0,1,2,3" — Ultralytics launches DDP
        workers=WORKERS,
        project=PROJECT_NAME,
        name=RUN_NAME,
        exist_ok=True,

        # ── Loss Weights ─────────────────────────────────────────────
        box=9.0,            # Tighter bounding boxes (default 7.5)
        cls=0.5,
        dfl=1.5,

        # ── HSV Color Jitter (YOLO built-in) ─────────────────────────
        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.4,

        # ── Geometric Augmentations (YOLO built-in) ───────────────────
        degrees=2.0,        # Conservative rotation for road slopes
        translate=0.1,
        scale=0.5,
        shear=0.0,          # Disabled — unnecessary for GSV perspective
        perspective=0.0,    # Disabled — GSV camera is roughly fixed

        # ── Flip ─────────────────────────────────────────────────────
        flipud=0.0,         # GSV is always upright
        fliplr=0.5,         # Poles appear on both sides of road

        # ── Composite Augmentations ───────────────────────────────────
        mosaic=1.0,         # Full mosaic — strong generalisation
        mixup=0.1,          # Light blending for background diversity
        copy_paste=0.2,     # Handles class imbalance

        # ── Random Erasing ────────────────────────────────────────────
        erasing=0.2,        # Moderate — avoids erasing entire thin poles

        # ── Mosaic Scheduling ─────────────────────────────────────────
        close_mosaic=10,    # Disable mosaic for final 10 epochs (fine-tune)

        # ── Optimizer & LR Schedule ───────────────────────────────────
        optimizer="AdamW",
        lr0=0.001,
        lrf=0.01,
        momentum=0.937,
        weight_decay=0.0005,
        warmup_epochs=3,
        warmup_momentum=0.8,
        cos_lr=True,        # Cosine annealing — smoother than step decay

        # ── Output & Validation ───────────────────────────────────────
        save=True,
        save_period=10,     # Checkpoint every 10 epochs
        plots=True,
        verbose=True,
        val=True,
        patience=20,        # Early stop if no improvement for 20 epochs

        # ── Multi-GPU Optimisations ───────────────────────────────────
        #sync_bn=True,       # Sync batch norm across GPUs
        amp=True,           # Mixed precision (faster + less VRAM)
        fraction=1.0,       # Use 100% of data per epoch
    )
    return results


# ─────────────────────────────────────────────────────────────────────
# STEP 2 — Evaluate on Test Set
# ─────────────────────────────────────────────────────────────────────

def evaluate_model(gpu_manager: GPUManager):
    best_weights = f"{PROJECT_NAME}/{RUN_NAME}/weights/best.pt"
    if not os.path.exists(best_weights):
        print(f"\n[WARN] best.pt not found at {best_weights}. Skipping evaluation.")
        return

    device_str  = gpu_manager.get_device_string()
    num_gpus    = max(len(gpu_manager.selected_gpus), 1)
    eval_batch  = BATCH_SIZE_PER_GPU * num_gpus

    print(f"\n[Step 2] Evaluating on test set...")
    print(f"  Weights : {best_weights}")
    print(f"  Device  : {device_str}\n")

    model   = YOLO(best_weights)
    metrics = model.val(
        data=str(DATA_YAML),
        split="test",
        imgsz=IMG_SIZE,
        batch=eval_batch,
        device=device_str,
        project=PROJECT_NAME,
        name=f"{RUN_NAME}_test_eval",
        verbose=True,
    )

    print("\n  ── Test Set Results ──────────────────────────────")
    print(f"  mAP@0.5      : {metrics.box.map50:.4f}")
    print(f"  mAP@0.5:0.95 : {metrics.box.map:.4f}")
    print(f"  Precision    : {metrics.box.mp:.4f}")
    print(f"  Recall       : {metrics.box.mr:.4f}")
    print("  ──────────────────────────────────────────────────")
    return metrics


# ─────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    print("=" * 70)
    print("  YOLOv8 Optimised Multi-GPU Pole & Wire Detection Training")
    print("  GPU-native augmentation | No CPU bottleneck | No disk I/O")
    print("=" * 70)

    # ── GPU Manager ───────────────────────────────────────────────────
    gpu_manager = GPUManager()
    gpu_manager.select_gpus(num_gpus=NUM_GPUS)

    # ── Verify required folders ────────────────────────────────────────
    required = {
        "train/images": TRAIN_IMG_DIR,
        "train/labels": TRAIN_LBL_DIR,
        "valid/images": VALID_IMG_DIR,
        "test/images" : TEST_IMG_DIR,
    }
    all_ok = True
    print()
    for name, path in required.items():
        exists = path.exists()
        print(f"  [{'OK' if exists else 'MISSING'}] {name}  →  {path}")
        if not exists:
            all_ok = False

    if not all_ok:
        raise FileNotFoundError(
            "\n[ERROR] One or more folders are missing.\n"
            "Expected structure:\n"
            "  /content/NEW_poledata/\n"
            "  ├── train/images/\n"
            "  ├── train/labels/\n"
            "  ├── valid/images/\n"
            "  ├── valid/labels/\n"
            "  ├── test/images/\n"
            "  ├── test/labels/\n"
            "  └── data.yaml\n"
        )

    print()
    fix_data_yaml()                 # Step 0 — Fix relative paths in data.yaml
    train_yolov8(gpu_manager)       # Step 1 — Multi-GPU training (GPU-native aug)
    evaluate_model(gpu_manager)     # Step 2 — Test set evaluation

    print("\n  ═══════════════════════════════════════════════════════════════")
    print("  ✓ Training Complete!")
    print("  ═══════════════════════════════════════════════════════════════")
    print(f"  Best weights : {PROJECT_NAME}/{RUN_NAME}/weights/best.pt")
    print(f"  Last weights : {PROJECT_NAME}/{RUN_NAME}/weights/last.pt")
    print(f"  Plots        : {PROJECT_NAME}/{RUN_NAME}/")
    print(f"\n  GPUs used    : {gpu_manager.get_device_string()}")
    print(f"  Batch total  : {BATCH_SIZE_PER_GPU * max(len(gpu_manager.selected_gpus or [1]), 1)}")
    print("\n  What changed vs. original:")
    print("  • Albumentations removed    → eliminated CPU bottleneck")
    print("  • No disk writes            → eliminated I/O bottleneck")
    print("  • All 4 A100s active        → 3–4× faster wall-clock time")
    print("  • YOLO aug runs on GPU      → same quality, zero overhead")
    print("  • imgsz=960                 → better thin-pole resolution")
    print("  • batch scales with GPUs    → max hardware utilisation")
    print("\n  To use your model:")
    print("    from ultralytics import YOLO")
    print(f"    model = YOLO('{PROJECT_NAME}/{RUN_NAME}/weights/best.pt')")
    print("    results = model.predict('image.jpg')")
    print("  ═══════════════════════════════════════════════════════════════\n")

  YOLOv8 Optimised Multi-GPU Pole & Wire Detection Training
  GPU-native augmentation | No CPU bottleneck | No disk I/O

╔══════════════════════════════════════════════════════╗
║ GPU DETECTION REPORT                                 ║
╚══════════════════════════════════════════════════════╝

  Total GPUs Available: 4

  GPU 0:
    ├─ Name  : NVIDIA A100-PCIE-40GB
    └─ Memory: 42.29 GB

  GPU 1:
    ├─ Name  : NVIDIA A100-PCIE-40GB
    └─ Memory: 42.29 GB

  GPU 2:
    ├─ Name  : NVIDIA A100-PCIE-40GB
    └─ Memory: 42.29 GB

  GPU 3:
    ├─ Name  : NVIDIA A100-PCIE-40GB
    └─ Memory: 42.29 GB

╔══════════════════════════════════════════════════════╗
║ GPU IDLE STATUS CHECK                                ║
╚══════════════════════════════════════════════════════╝

  GPU 0: 0.0% used [IDLE ✓]
  GPU 1: 0.0% used [IDLE ✓]
  GPU 2: 0.0% used [IDLE ✓]
  GPU 3: 0.0% used [IDLE ✓]

╔══════════════════════════════════════════════════════╗
║ GPU SELECTION FOR TRAINING                          

/home/maska/.conda/envs/torch_env/lib/python3.10/site-packages/torch/utils/data/dataloader.py:617: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 4.0±2.8 MB/s, size: 63.1 KB)
val: Scanning /nfs/storage1/home/maska/NEW_poledata/valid/labels.cache... 1399 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1399/1399 76.2Mit/s 0.0s


/home/maska/.conda/envs/torch_env/lib/python3.10/site-packages/torch/utils/data/dataloader.py:617: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
/home/maska/.conda/envs/torch_env/lib/python3.10/site-packages/torch/utils/data/dataloader.py:617: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
/home/maska/.conda/envs/torch_env/lib/python

optimizer: AdamW(lr=0.001, momentum=0.937) with parameter groups 97 weight(decay=0.0), 104 weight(decay=0.0005), 103 bias(decay=0.0)


/home/maska/.conda/envs/torch_env/lib/python3.10/site-packages/torch/utils/data/dataloader.py:617: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
/home/maska/.conda/envs/torch_env/lib/python3.10/site-packages/torch/utils/data/dataloader.py:617: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
/home/maska/.conda/envs/torch_env/lib/pyth

Plotting labels to /nfs/storage1/home/maska/gsv_pole_detection1/exp1_optimized_multiGPU/labels.jpg... 
Image sizes 640 train, 640 val
Using 32 dataloader workers
Logging results to /nfs/storage1/home/maska/gsv_pole_detection1/exp1_optimized_multiGPU
Starting training for 100 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      1/100      9.54G      2.131      2.019      1.572         12        640: 100% ━━━━━━━━━━━━ 134/134 1.2s/it 2:341.3ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 11/11 1.4it/s 7.8s0.7s
                   all       1399       1586      0.355      0.508      0.311      0.127

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      2/100        11G      2.041      1.622      1.537         13        640: 100% ━━━━━━━━━━━━ 134/134 1.1s/it 2:231.0ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 

/home/maska/.conda/envs/torch_env/lib/python3.10/site-packages/torch/utils/data/dataloader.py:617: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 15/15 1.1it/s 13.5s0.5s
                   all        905       1010      0.872       0.83      0.901      0.536
Speed: 3.7ms preprocess, 4.2ms inference, 0.0ms loss, 1.2ms postprocess per image
Results saved to /nfs/storage1/home/maska/gsv_pole_detection1/exp1_optimized_multiGPU_test_eval

  ── Test Set Results ──────────────────────────────
  mAP@0.5      : 0.9014
  mAP@0.5:0.95 : 0.5355
  Precision    : 0.8722
  Recall       : 0.8297
  ──────────────────────────────────────────────────

  ═══════════════════════════════════════════════════════════════
  ✓ Training Complete!
  ═══════════════════════════════════════════════════════════════
  Best weights : /home/maska/gsv_pole_detection1/exp1_optimized_multiGPU/weights/best.pt
  Last weights : /home/maska/gsv_pole_detection1/exp1_optimized_multiGPU/weights/last.pt
  Plots        : /home/maska/gsv_pole_detection1/exp1_optimized_

In [7]:
# ── Install dependencies (run this cell first in Colab or server) ──────────────
# !pip install ultralytics opencv-python pyyaml torch torchvision -q

import os
import yaml
import torch
from pathlib import Path
from typing import List, Optional
from ultralytics import YOLO

# ─────────────────────────────────────────────────────────────────────
# MULTI-GPU CONFIGURATION & UTILITIES
# ─────────────────────────────────────────────────────────────────────

class GPUManager:
    """
    Manages GPU detection, idle GPU identification, and multi-GPU allocation.
    Optimizes for server deployments with multiple GPUs.
    """

    def __init__(self):
        self.available_gpus = self._detect_gpus()
        self.idle_gpus      = self._identify_idle_gpus()
        self.selected_gpus  = None

    def _detect_gpus(self) -> List[int]:
        """Detect all available GPUs on the system."""
        if not torch.cuda.is_available():
            print("[WARN] CUDA not available. Training will run on CPU.")
            return []

        num_gpus = torch.cuda.device_count()
        gpus     = list(range(num_gpus))

        print(f"\n╔══════════════════════════════════════════════════════╗")
        print(f"║ GPU DETECTION REPORT                                 ║")
        print(f"╚══════════════════════════════════════════════════════╝")
        print(f"\n  Total GPUs Available: {num_gpus}")

        for gpu_id in gpus:
            gpu_name   = torch.cuda.get_device_name(gpu_id)
            gpu_memory = torch.cuda.get_device_properties(gpu_id).total_memory / 1e9
            print(f"\n  GPU {gpu_id}:")
            print(f"    ├─ Name  : {gpu_name}")
            print(f"    └─ Memory: {gpu_memory:.2f} GB")

        return gpus

    def _identify_idle_gpus(self) -> List[int]:
        """
        Identify idle GPUs by checking memory usage.
        A GPU is considered idle if memory usage < 10% of total.
        """
        if not self.available_gpus:
            return []

        idle_gpus = []
        print(f"\n╔══════════════════════════════════════════════════════╗")
        print(f"║ GPU IDLE STATUS CHECK                                ║")
        print(f"╚══════════════════════════════════════════════════════╝\n")

        for gpu_id in self.available_gpus:
            try:
                torch.cuda.reset_peak_memory_stats(gpu_id)
                current_memory = torch.cuda.memory_allocated(gpu_id) / 1e9
                total_memory   = torch.cuda.get_device_properties(gpu_id).total_memory / 1e9
                memory_percent = (current_memory / total_memory) * 100

                status = "IDLE ✓" if memory_percent < 10 else "IN USE ✗"
                print(f"  GPU {gpu_id}: {memory_percent:.1f}% used [{status}]")

                if memory_percent < 10:
                    idle_gpus.append(gpu_id)

            except Exception as e:
                print(f"  [ERROR] Could not check GPU {gpu_id}: {str(e)}")

        return idle_gpus

    def select_gpus(self, num_gpus: Optional[int] = None) -> List[int]:
        """
        Select GPUs for training.

        Args:
            num_gpus: Number of GPUs to use. If None, use all idle GPUs.

        Returns:
            List of GPU IDs to use for training.
        """
        if not self.available_gpus:
            print("\n[ERROR] No GPUs available. CPU training will be very slow.")
            self.selected_gpus = []
            return []

        if not self.idle_gpus:
            print(f"\n[WARN] No idle GPUs detected. Using first GPU anyway.")
            self.selected_gpus = [self.available_gpus[0]]
        else:
            self.selected_gpus = self.idle_gpus if num_gpus is None else self.idle_gpus[:num_gpus]

        print(f"\n╔══════════════════════════════════════════════════════╗")
        print(f"║ GPU SELECTION FOR TRAINING                           ║")
        print(f"╚══════════════════════════════════════════════════════╝")
        print(f"\n  Selected GPUs : {self.selected_gpus}")
        print(f"  Count         : {len(self.selected_gpus)}")

        return self.selected_gpus

    def get_device_string(self) -> str:
        """
        Return device string for YOLO training.

        Returns:
            Comma-separated GPU IDs (e.g. "0,1,2,3") or "cpu".
        """
        if not self.selected_gpus:
            return "cpu"
        return ",".join(map(str, self.selected_gpus))


# ─────────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────────

HOME_DIR = Path.home()
BASE_DIR        = HOME_DIR / "NEW_poledata"

TRAIN_IMG_DIR   = BASE_DIR / "train" / "images"
TRAIN_LBL_DIR   = BASE_DIR / "train" / "labels"
VALID_IMG_DIR   = BASE_DIR / "valid" / "images"
VALID_LBL_DIR   = BASE_DIR / "valid" / "labels"
TEST_IMG_DIR    = BASE_DIR / "test"  / "images"
TEST_LBL_DIR    = BASE_DIR / "test"  / "labels"



# data.yaml
DATA_YAML   = BASE_DIR / "data.yaml"



# YOLOv8 settings — OPTIMIZED
MODEL_WEIGHTS   = "yolov8l.pt"        
PROJECT_NAME    = str(HOME_DIR / "gsv_pole_detection1")
RUN_NAME              = "exp1_optimized_multiGPU960imgsize"
EPOCHS                = 100
IMG_SIZE              = 960          # Better for thin vertical objects
BATCH_SIZE_PER_GPU    = 16           # Scales with GPU count; increase if memory allows
NUM_GPUS              = None         # None = auto-select all idle GPUs
WORKERS               = 8           # Increased for A100 multi-GPU setup


# ─────────────────────────────────────────────────────────────────────
# STEP 0 — Auto-fix data.yaml to use absolute paths
# ─────────────────────────────────────────────────────────────────────

def fix_data_yaml():
    """
    Rewrites data.yaml with absolute paths so YOLOv8 can find the data
    regardless of the working directory.
    """
    if not DATA_YAML.exists():
        raise FileNotFoundError(f"data.yaml not found at: {DATA_YAML}")

    with open(DATA_YAML, "r") as f:
        data = yaml.safe_load(f)

    data["train"] = str(TRAIN_IMG_DIR)
    data["val"]   = str(VALID_IMG_DIR)
    data["test"]  = str(TEST_IMG_DIR)

    with open(DATA_YAML, "w") as f:
        yaml.dump(data, f, default_flow_style=False)

    print(f"[Step 0] data.yaml fixed with absolute paths:")
    print(f"  train → {TRAIN_IMG_DIR}")
    print(f"  val   → {VALID_IMG_DIR}")
    print(f"  test  → {TEST_IMG_DIR}")


# ─────────────────────────────────────────────────────────────────────
# STEP 1 — YOLOv8 Training (GPU-native, on-the-fly augmentation)
# ─────────────────────────────────────────────────────────────────────

def train_yolov8(gpu_manager: GPUManager):
    """
    Train YOLOv8 directly on the original dataset.

    All augmentation is handled on-the-fly by YOLO's built-in pipeline —
    no CPU pre-processing, no disk I/O overhead, full GPU utilisation.

    KEY DECISIONS:
      • No Albumentations: eliminated CPU bottleneck & disk writes
      • device="0,1,2,3": DDP automatically engaged by Ultralytics
      • batch = BATCH_SIZE_PER_GPU × num_gpus: maximises throughput
      • imgsz=960: critical for detecting thin poles
      • box=9.0: forces tighter bounding boxes
      • cos_lr=True: smoother decay schedule
      • sync_bn=True: stable batch-norm across GPUs
      • amp=True: mixed-precision for memory efficiency
    """
    model = YOLO(MODEL_WEIGHTS)

    num_gpus     = max(len(gpu_manager.selected_gpus), 1) if gpu_manager.selected_gpus else 1
    total_batch  = BATCH_SIZE_PER_GPU * num_gpus
    device_str   = gpu_manager.get_device_string()

    print(f"\n[Step 1] Starting YOLOv8 Multi-GPU Training")
    print(f"  Model      : {MODEL_WEIGHTS}")
    print(f"  Epochs     : {EPOCHS}")
    print(f"  Image size : {IMG_SIZE}px")
    print(f"  GPUs       : {device_str}")
    print(f"  Batch size : {total_batch}  ({BATCH_SIZE_PER_GPU} × {num_gpus} GPUs)")
    print(f"  Workers    : {WORKERS}")
    print(f"  Data YAML  : {DATA_YAML}\n")

    results = model.train(
        data=str(DATA_YAML),        # ← original dataset, no augmented copies
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=total_batch,
        device=device_str,          # e.g. "0,1,2,3" — Ultralytics launches DDP
        workers=WORKERS,
        project=PROJECT_NAME,
        name=RUN_NAME,
        exist_ok=True,

        # ── Loss Weights ─────────────────────────────────────────────
        box=9.0,            # Tighter bounding boxes (default 7.5)
        cls=0.5,
        dfl=1.5,

        # ── HSV Color Jitter (YOLO built-in) ─────────────────────────
        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.4,

        # ── Geometric Augmentations (YOLO built-in) ───────────────────
        degrees=2.0,        # Conservative rotation for road slopes
        translate=0.1,
        scale=0.5,
        shear=0.0,          # Disabled — unnecessary for GSV perspective
        perspective=0.0,    # Disabled — GSV camera is roughly fixed

        # ── Flip ─────────────────────────────────────────────────────
        flipud=0.0,         # GSV is always upright
        fliplr=0.5,         # Poles appear on both sides of road

        # ── Composite Augmentations ───────────────────────────────────
        mosaic=1.0,         # Full mosaic — strong generalisation
        mixup=0.1,          # Light blending for background diversity
        copy_paste=0.2,     # Handles class imbalance

        # ── Random Erasing ────────────────────────────────────────────
        erasing=0.2,        # Moderate — avoids erasing entire thin poles

        # ── Mosaic Scheduling ─────────────────────────────────────────
        close_mosaic=10,    # Disable mosaic for final 10 epochs (fine-tune)

        # ── Optimizer & LR Schedule ───────────────────────────────────
        optimizer="AdamW",
        lr0=0.001,
        lrf=0.01,
        momentum=0.937,
        weight_decay=0.0005,
        warmup_epochs=3,
        warmup_momentum=0.8,
        cos_lr=True,        # Cosine annealing — smoother than step decay

        # ── Output & Validation ───────────────────────────────────────
        save=True,
        save_period=10,     # Checkpoint every 10 epochs
        plots=True,
        verbose=True,
        val=True,
        patience=20,        # Early stop if no improvement for 20 epochs

        # ── Multi-GPU Optimisations ───────────────────────────────────
        #sync_bn=True,       # Sync batch norm across GPUs
        amp=True,           # Mixed precision (faster + less VRAM)
        fraction=1.0,       # Use 100% of data per epoch
    )
    return results


# ─────────────────────────────────────────────────────────────────────
# STEP 2 — Evaluate on Test Set
# ─────────────────────────────────────────────────────────────────────

def evaluate_model(gpu_manager: GPUManager):
    best_weights = f"{PROJECT_NAME}/{RUN_NAME}/weights/best.pt"
    if not os.path.exists(best_weights):
        print(f"\n[WARN] best.pt not found at {best_weights}. Skipping evaluation.")
        return

    device_str  = gpu_manager.get_device_string()
    num_gpus    = max(len(gpu_manager.selected_gpus), 1)
    eval_batch  = BATCH_SIZE_PER_GPU * num_gpus

    print(f"\n[Step 2] Evaluating on test set...")
    print(f"  Weights : {best_weights}")
    print(f"  Device  : {device_str}\n")

    model   = YOLO(best_weights)
    metrics = model.val(
        data=str(DATA_YAML),
        split="test",
        imgsz=IMG_SIZE,
        batch=eval_batch,
        device=device_str,
        project=PROJECT_NAME,
        name=f"{RUN_NAME}_test_eval",
        verbose=True,
    )

    print("\n  ── Test Set Results ──────────────────────────────")
    print(f"  mAP@0.5      : {metrics.box.map50:.4f}")
    print(f"  mAP@0.5:0.95 : {metrics.box.map:.4f}")
    print(f"  Precision    : {metrics.box.mp:.4f}")
    print(f"  Recall       : {metrics.box.mr:.4f}")
    print("  ──────────────────────────────────────────────────")
    return metrics


# ─────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    print("=" * 70)
    print("  YOLOv8 Optimised Multi-GPU Pole & Wire Detection Training")
    print("  GPU-native augmentation | No CPU bottleneck | No disk I/O")
    print("=" * 70)

    # ── GPU Manager ───────────────────────────────────────────────────
    gpu_manager = GPUManager()
    gpu_manager.select_gpus(num_gpus=NUM_GPUS)

    # ── Verify required folders ────────────────────────────────────────
    required = {
        "train/images": TRAIN_IMG_DIR,
        "train/labels": TRAIN_LBL_DIR,
        "valid/images": VALID_IMG_DIR,
        "test/images" : TEST_IMG_DIR,
    }
    all_ok = True
    print()
    for name, path in required.items():
        exists = path.exists()
        print(f"  [{'OK' if exists else 'MISSING'}] {name}  →  {path}")
        if not exists:
            all_ok = False

    if not all_ok:
        raise FileNotFoundError(
            "\n[ERROR] One or more folders are missing.\n"
            "Expected structure:\n"
            "  /content/NEW_poledata/\n"
            "  ├── train/images/\n"
            "  ├── train/labels/\n"
            "  ├── valid/images/\n"
            "  ├── valid/labels/\n"
            "  ├── test/images/\n"
            "  ├── test/labels/\n"
            "  └── data.yaml\n"
        )

    print()
    fix_data_yaml()                 # Step 0 — Fix relative paths in data.yaml
    train_yolov8(gpu_manager)       # Step 1 — Multi-GPU training (GPU-native aug)
    evaluate_model(gpu_manager)     # Step 2 — Test set evaluation

    print("\n  ═══════════════════════════════════════════════════════════════")
    print("  ✓ Training Complete!")
    print("  ═══════════════════════════════════════════════════════════════")
    print(f"  Best weights : {PROJECT_NAME}/{RUN_NAME}/weights/best.pt")
    print(f"  Last weights : {PROJECT_NAME}/{RUN_NAME}/weights/last.pt")
    print(f"  Plots        : {PROJECT_NAME}/{RUN_NAME}/")
    print(f"\n  GPUs used    : {gpu_manager.get_device_string()}")
    print(f"  Batch total  : {BATCH_SIZE_PER_GPU * max(len(gpu_manager.selected_gpus or [1]), 1)}")
    print("\n  What changed vs. original:")
    print("  • Albumentations removed    → eliminated CPU bottleneck")
    print("  • No disk writes            → eliminated I/O bottleneck")
    print("  • All 4 A100s active        → 3–4× faster wall-clock time")
    print("  • YOLO aug runs on GPU      → same quality, zero overhead")
    print("  • imgsz=960                 → better thin-pole resolution")
    print("  • batch scales with GPUs    → max hardware utilisation")
    print("\n  To use your model:")
    print("    from ultralytics import YOLO")
    print(f"    model = YOLO('{PROJECT_NAME}/{RUN_NAME}/weights/best.pt')")
    print("    results = model.predict('image.jpg')")
    print("  ═══════════════════════════════════════════════════════════════\n")

  YOLOv8 Optimised Multi-GPU Pole & Wire Detection Training
  GPU-native augmentation | No CPU bottleneck | No disk I/O

╔══════════════════════════════════════════════════════╗
║ GPU DETECTION REPORT                                 ║
╚══════════════════════════════════════════════════════╝

  Total GPUs Available: 4

  GPU 0:
    ├─ Name  : NVIDIA A100-PCIE-40GB
    └─ Memory: 42.29 GB

  GPU 1:
    ├─ Name  : NVIDIA A100-PCIE-40GB
    └─ Memory: 42.29 GB

  GPU 2:
    ├─ Name  : NVIDIA A100-PCIE-40GB
    └─ Memory: 42.29 GB

  GPU 3:
    ├─ Name  : NVIDIA A100-PCIE-40GB
    └─ Memory: 42.29 GB

╔══════════════════════════════════════════════════════╗
║ GPU IDLE STATUS CHECK                                ║
╚══════════════════════════════════════════════════════╝

  GPU 0: 0.0% used [IDLE ✓]
  GPU 1: 0.0% used [IDLE ✓]
  GPU 2: 0.0% used [IDLE ✓]
  GPU 3: 0.0% used [IDLE ✓]

╔══════════════════════════════════════════════════════╗
║ GPU SELECTION FOR TRAINING                          

/home/maska/.conda/envs/torch_env/lib/python3.10/site-packages/torch/utils/data/dataloader.py:617: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 69.4±88.9 MB/s, size: 63.1 KB)
val: Scanning /nfs/storage1/home/maska/NEW_poledata/valid/labels.cache... 1399 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1399/1399 75.2Mit/s 0.0s


/home/maska/.conda/envs/torch_env/lib/python3.10/site-packages/torch/utils/data/dataloader.py:617: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
/home/maska/.conda/envs/torch_env/lib/python3.10/site-packages/torch/utils/data/dataloader.py:617: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
/home/maska/.conda/envs/torch_env/lib/python

optimizer: AdamW(lr=0.001, momentum=0.937) with parameter groups 97 weight(decay=0.0), 104 weight(decay=0.0005), 103 bias(decay=0.0)
Plotting labels to /nfs/storage1/home/maska/gsv_pole_detection1/exp1_optimized_multiGPU960imgsize/labels.jpg... 
Image sizes 960 train, 960 val
Using 32 dataloader workers
Logging results to /nfs/storage1/home/maska/gsv_pole_detection1/exp1_optimized_multiGPU960imgsize
Starting training for 100 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      1/100      20.7G      2.183      2.534       1.67         12        960: 100% ━━━━━━━━━━━━ 134/134 2.1s/it 4:441.7ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 11/11 1.1s/it 12.5s1.2s
                   all       1399       1586     0.0999      0.267     0.0808     0.0258

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      2/100      20.3G       2.06       1.77      1.595    

/home/maska/.conda/envs/torch_env/lib/python3.10/site-packages/torch/utils/data/dataloader.py:617: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
/home/maska/.conda/envs/torch_env/lib/python3.10/site-packages/torch/utils/data/dataloader.py:617: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
/home/maska/.conda/envs/torch_env/lib/python


      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     91/100      20.3G      1.363     0.8193      1.228          7        960: 100% ━━━━━━━━━━━━ 134/134 2.0s/it 4:251.6ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 11/11 1.1s/it 12.0s1.2s
                   all       1399       1586      0.795      0.792      0.863      0.482
EarlyStopping: Training stopped early as no improvement observed in last 20 epochs. Best results observed at epoch 71, best model saved as best.pt.
To update EarlyStopping(patience=20) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.

91 epochs completed in 6.852 hours.
Optimizer stripped from /nfs/storage1/home/maska/gsv_pole_detection1/exp1_optimized_multiGPU960imgsize/weights/last.pt, 87.7MB
Optimizer stripped from /nfs/storage1/home/maska/gsv_pole_detection1/exp1_optimized_multiGPU960imgsize/weights/best.pt, 87.7MB

Validat

/home/maska/.conda/envs/torch_env/lib/python3.10/site-packages/torch/utils/data/dataloader.py:617: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 15/15 1.7s/it 26.1s0.9s
                   all        905       1010       0.88       0.84      0.908      0.544
Speed: 9.7ms preprocess, 8.6ms inference, 0.0ms loss, 1.4ms postprocess per image
Results saved to /nfs/storage1/home/maska/gsv_pole_detection1/exp1_optimized_multiGPU960imgsize_test_eval

  ── Test Set Results ──────────────────────────────
  mAP@0.5      : 0.9077
  mAP@0.5:0.95 : 0.5435
  Precision    : 0.8804
  Recall       : 0.8396
  ──────────────────────────────────────────────────

  ═══════════════════════════════════════════════════════════════
  ✓ Training Complete!
  ═══════════════════════════════════════════════════════════════
  Best weights : /home/maska/gsv_pole_detection1/exp1_optimized_multiGPU960imgsize/weights/best.pt
  Last weights : /home/maska/gsv_pole_detection1/exp1_optimized_multiGPU960imgsize/weights/last.pt
  Plots        : /home/maska/gsv_p

In [8]:
# # ── Install dependencies (run this cell first in Colab) ──────────────
# # !pip install ultralytics albumentations opencv-python pyyaml -q

# import os
# import cv2
# import random
# import shutil
# import yaml
# import numpy as np
# from pathlib import Path
# import albumentations as A
# from ultralytics import YOLO

# # ─────────────────────────────────────────────────────────────────────
# # CONFIG — OPTIMIZED FOR UTILITY POLE DETECTION
# # ─────────────────────────────────────────────────────────────────────

# #  FIX 1: Correct home dir — confirmed from traceback: /home/jovyan/
# HOME_DIR = Path.home()
# BASE_DIR        = HOME_DIR / "NEW_poledata"

# TRAIN_IMG_DIR   = BASE_DIR / "train" / "images"
# TRAIN_LBL_DIR   = BASE_DIR / "train" / "labels"
# VALID_IMG_DIR   = BASE_DIR / "valid" / "images"
# VALID_LBL_DIR   = BASE_DIR / "valid" / "labels"
# TEST_IMG_DIR    = BASE_DIR / "test"  / "images"
# TEST_LBL_DIR    = BASE_DIR / "test"  / "labels"

# # Albumentations output (augmented training images)
# AUG_TRAIN_IMG   = BASE_DIR / "train_augmented" / "images"
# AUG_TRAIN_LBL   = BASE_DIR / "train_augmented" / "labels"

# # data.yaml
# ORIGINAL_YAML   = BASE_DIR / "data.yaml"
# AUG_YAML        = BASE_DIR / "data_augmented.yaml"


# # YOLOv8 settings — OPTIMIZED
# MODEL_WEIGHTS   = "yolov8l.pt"        # UPGRADED: yolov8n → yolov8l (better for thin objects)
# PROJECT_NAME    = str(HOME_DIR / "gsv_pole_detection1")
# RUN_NAME        = "exp1_optimized"
# EPOCHS          = 100                  # Keep at 100 as requested
# IMG_SIZE        = 640                  # UPGRADED: 640  (critical for thin poles)
# BATCH_SIZE      = 16                    # REDUCED: 16 → 8 (to accommodate higher resolution)
# DEVICE          = 0                    # 0 = GPU (T4), "cpu" = CPU
# WORKERS         = 2                    # keep low on Colab to avoid DataLoader errors
# AUG_COPIES      = 1                    # augmented copies per image → 4x dataset size


# # ─────────────────────────────────────────────────────────────────────
# # STEP 0 — Auto-fix data.yaml to use absolute paths
# # ─────────────────────────────────────────────────────────────────────

# def fix_data_yaml():
#     """
#     Your original data.yaml uses relative paths like ../train/images.
#     This function rewrites it with absolute Colab paths so YOLOv8
#     can find your data correctly.
#     """
#     if not ORIGINAL_YAML.exists():
#         raise FileNotFoundError(f"data.yaml not found at: {ORIGINAL_YAML}")

#     with open(ORIGINAL_YAML, "r") as f:
#         data = yaml.safe_load(f)

#     data["train"] = str(TRAIN_IMG_DIR)
#     data["val"]   = str(VALID_IMG_DIR)
#     data["test"]  = str(TEST_IMG_DIR)

#     # Overwrite in place with fixed absolute paths
#     with open(ORIGINAL_YAML, "w") as f:
#         yaml.dump(data, f, default_flow_style=False)

#     print(f"[Step 0] data.yaml fixed with absolute paths:")
#     print(f"  train → {TRAIN_IMG_DIR}")
#     print(f"  val   → {VALID_IMG_DIR}")
#     print(f"  test  → {TEST_IMG_DIR}")


# # ─────────────────────────────────────────────────────────────────────
# # STEP 1 — OPTIMIZED ALBUMENTATIONS PIPELINE FOR THIN POLES
# # ─────────────────────────────────────────────────────────────────────

# def get_albumentations_pipeline():
#     """
#     OPTIMIZED Augmentation pipeline for electrical pole and wire detection.
    
#     CHANGES FROM ORIGINAL:
#     - Reduced blur probabilities (MotionBlur p=0.1, GaussianBlur p=0.1)
#     - Reduced weather probabilities (Fog p=0.05, Rain p=0.05) - was p=0.15, p=0.1
#     - Kept RandomShadow and brightness/contrast (realistic for GSV)
#     - Removed RandomSunFlare (too aggressive for pole edges)
#     - Kept CLAHE for contrast enhancement
    
#     This balances domain-specific needs (GSV weather) with practical 
#     requirements (preserving thin pole structure).
#     """
#     return A.Compose(
#         [
#             # Lighting & Contrast (preserved - realistic for GSV)
#             A.RandomBrightnessContrast(p=0.5),
#             A.CLAHE(clip_limit=4.0, p=0.3),

#             # Noise
#             A.GaussNoise(p=0.3),

#             # Blur - REDUCED (was p=0.2, now p=0.1 to preserve pole edges)
#             A.MotionBlur(blur_limit=5, p=0.1),
#             A.GaussianBlur(blur_limit=3, p=0.1),

#             # Weather effects - REDUCED (was p=0.15/0.1, now p=0.05)
#             A.RandomFog(
#                 fog_coef_range=(0.1, 0.3),
#                 alpha_coef=0.1,
#                 p=0.05  # REDUCED: was 0.15
#             ),
#             A.RandomRain(
#                 slant_range=(-10, 10),
#                 drop_length=10,
#                 drop_width=1,
#                 brightness_coefficient=0.9,
#                 p=0.05  # REDUCED: was 0.1
#             ),
#             # REMOVED: RandomSunFlare (too harsh for thin objects)
            
#             A.RandomShadow(
#                 shadow_roi=(0, 0.5, 1, 1),  # lower half (building shadows)
#                 num_shadows_limit=(1, 2),
#                 shadow_dimension=5,
#                 p=0.2  # KEPT: realistic for street view
#             ),
#         ],
#         bbox_params=A.BboxParams(
#             format="yolo",                  # cx cy w h (normalized 0-1)
#             label_fields=["class_labels"],
#             min_visibility=0.3,             # drop bbox if <30% visible
#         ),
#     )


# # ─────────────────────────────────────────────────────────────────────
# # HELPERS
# # ─────────────────────────────────────────────────────────────────────

# def read_yolo_labels(label_path):
#     bboxes, class_labels = [], []
#     if not os.path.exists(label_path):
#         return bboxes, class_labels
#     with open(label_path, "r") as f:
#         for line in f:
#             parts = line.strip().split()
#             if len(parts) == 5:
#                 cls = int(parts[0])
#                 cx, cy, w, h = map(float, parts[1:])
#                 bboxes.append([cx, cy, w, h])
#                 class_labels.append(cls)
#     return bboxes, class_labels


# def write_yolo_labels(label_path, bboxes, class_labels):
#     with open(label_path, "w") as f:
#         for cls, bbox in zip(class_labels, bboxes):
#             f.write(f"{cls} {bbox[0]:.6f} {bbox[1]:.6f} {bbox[2]:.6f} {bbox[3]:.6f}\n")


# # ─────────────────────────────────────────────────────────────────────
# # STEP 1 — Apply Albumentations to training set
# # ─────────────────────────────────────────────────────────────────────

# def apply_albumentations_to_dataset(seed=42):
#     random.seed(seed)
#     np.random.seed(seed)

#     os.makedirs(AUG_TRAIN_IMG, exist_ok=True)
#     os.makedirs(AUG_TRAIN_LBL, exist_ok=True)

#     pipeline    = get_albumentations_pipeline()
#     image_paths = (
#         sorted(Path(TRAIN_IMG_DIR).glob("*.jpg"))
#         + sorted(Path(TRAIN_IMG_DIR).glob("*.jpeg"))
#         + sorted(Path(TRAIN_IMG_DIR).glob("*.png"))
#     )

#     print(f"\n[Step 1] Found {len(image_paths)} training images")
#     print(f"         Generating {AUG_COPIES} augmented copies each "
#           f"→ {len(image_paths) * (AUG_COPIES + 1)} total images\n")

#     for idx, img_path in enumerate(image_paths):
#         img_path = Path(img_path)
#         stem     = img_path.stem
#         ext      = img_path.suffix
#         lbl_path = TRAIN_LBL_DIR / (stem + ".txt")

#         bboxes, class_labels = read_yolo_labels(str(lbl_path))

#         image = cv2.imread(str(img_path))
#         if image is None:
#             print(f"  [WARN] Could not read {img_path.name}, skipping.")
#             continue
#         image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

#         # Save original into augmented folder
#         cv2.imwrite(
#             str(AUG_TRAIN_IMG / (stem + ext)),
#             cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
#         )
#         if lbl_path.exists():
#             shutil.copy(str(lbl_path), str(AUG_TRAIN_LBL / (stem + ".txt")))
#         else:
#             open(str(AUG_TRAIN_LBL / (stem + ".txt")), "w").close()

#         # Save augmented copies
#         for i in range(AUG_COPIES):
#             try:
#                 result   = pipeline(image=image, bboxes=bboxes, class_labels=class_labels)
#                 aug_stem = f"{stem}_aug{i + 1}"
#                 cv2.imwrite(
#                     str(AUG_TRAIN_IMG / (aug_stem + ext)),
#                     cv2.cvtColor(result["image"], cv2.COLOR_RGB2BGR)
#                 )
#                 write_yolo_labels(
#                     str(AUG_TRAIN_LBL / (aug_stem + ".txt")),
#                     result["bboxes"],
#                     result["class_labels"]
#                 )
#             except Exception as e:
#                 # Silently skip augmentation failures
#                 pass

#         # Progress every 50 images
#         if (idx + 1) % 50 == 0:
#             print(f"  Processed {idx + 1}/{len(image_paths)} images...")

#     total = len(list(AUG_TRAIN_IMG.glob("*.*")))
#     print(f"\n  Done. Total images in train_augmented/images: {total}")


# # ─────────────────────────────────────────────────────────────────────
# # STEP 2 — Create data_augmented.yaml
# # ─────────────────────────────────────────────────────────────────────

# def create_augmented_yaml():
#     """
#     Creates data_augmented.yaml with:
#       train → train_augmented/images  (Albumentations output)
#       val   → valid/images            (original, unchanged)
#       test  → test/images             (original, unchanged)
#     """
#     with open(ORIGINAL_YAML, "r") as f:
#         data = yaml.safe_load(f)

#     data["train"] = str(AUG_TRAIN_IMG)
#     data["val"]   = str(VALID_IMG_DIR)
#     data["test"]  = str(TEST_IMG_DIR)

#     with open(AUG_YAML, "w") as f:
#         yaml.dump(data, f, default_flow_style=False)

#     print(f"\n[Step 2] data_augmented.yaml created at:\n  {AUG_YAML}")
#     print(f"  train → {AUG_TRAIN_IMG}")
#     print(f"  val   → {VALID_IMG_DIR}")
#     print(f"  test  → {TEST_IMG_DIR}")


# # ─────────────────────────────────────────────────────────────────────
# # STEP 3 — YOLOv8 Training — OPTIMIZED FOR POLE PRECISION
# # ─────────────────────────────────────────────────────────────────────

# def train_yolov8():
#     """
#     OPTIMIZED training configuration for utility pole detection.
    
#     KEY CHANGES FROM ORIGINAL:
#     1. box=9.0 (increased from default 7.5) - FORCES TIGHTER BOUNDING BOXES
#     2. IMG_SIZE=960 (from 640) - Better for thin vertical objects
#     3. BATCH_SIZE=8 (from 16) - Accommodate higher resolution
#     4. YOLOv8-L (from yolov8m) - Larger model for better feature learning
#     5. Reduced blur/weather in Albumentations
#     6. mosaic=0.8 (from 1.0) - Slightly safer for small objects
#     7. copy_paste=0.3 (kept from original) - Balance class imbalance
#     8. erasing=0.25 (from 0.3) - Avoid removing entire thin poles
#     9. cos_lr=True - Smoother learning rate schedule
#     """
#     model = YOLO(MODEL_WEIGHTS)

#     print(f"\n[Step 3] Starting YOLOv8 OPTIMIZED training...")
#     print(f"  Model    : {MODEL_WEIGHTS} (UPGRADED from yolov8m)")
#     print(f"  Epochs   : {EPOCHS}")
#     print(f"  Img size : {IMG_SIZE} (UPGRADED from 640)")
#     print(f"  Batch    : {BATCH_SIZE} (REDUCED to fit higher resolution)\n")

#     results = model.train(
#         data=str(AUG_YAML),
#         epochs=EPOCHS,
#         imgsz=IMG_SIZE,
#         batch=BATCH_SIZE,
#         device=DEVICE,
#         workers=WORKERS,
#         project=PROJECT_NAME,
#         name=RUN_NAME,
#         exist_ok=True,

#         # ── LOSS WEIGHTS (OPTIMIZED FOR BOX PRECISION) ─────────────
#         # CRITICAL: Increased box weight to force tighter bounding boxes
#         box=9.0,            # INCREASED: was default 7.5 → 9.0 (+20%)
#         cls=0.5,            # KEPT: classification loss
#         dfl=1.5,            # KEPT: distribution focal loss

#         # ── HSV Color Jitter ─────────────────────────────────────
#         hsv_h=0.015,        # Mild hue shift for lighting variation
#         hsv_s=0.7,          # Saturation variation
#         hsv_v=0.5,          # Brightness variation (increased from default)

#         # ── Geometric Augmentations ──────────────────────────────
#         degrees=4.0,        # Conservative rotation for road slopes
#         translate=0.1,      # Horizontal/vertical translation
#         scale=0.5,          # Scale variation
#         shear=1.5,          # Conservative shear for vehicle-mounted camera
#         perspective=0.0003, # Very light perspective distortion

#         # ── Flip ─────────────────────────────────────────────────
#         flipud=0.0,         # DISABLED: GSV is always upright
#         fliplr=0.5,         # Poles appear on both sides of road

#         # ── Composite Augmentations ──────────────────────────────
#         mosaic=0.8,         # REDUCED: was 1.0 → 0.8 (safer for small objects)
#         mixup=0.12,         # Blends 2 images for background diversity
#         copy_paste=0.3,     # Handles class imbalance

#         # ── Occlusion (Random Erasing) ──────────────────────────
#         erasing=0.25,       # REDUCED: was 0.3 → 0.25 (avoid removing thin poles)

#         # ── Mosaic Scheduling ────────────────────────────────────
#         close_mosaic=10,    # Disable mosaic in final 10 epochs

#         # ── Learning Rate Schedule ───────────────────────────────
#         optimizer="AdamW",  # Adaptive moment estimation optimizer
#         lr0=0.001,          # Initial learning rate
#         lrf=0.01,           # Final learning rate ratio
#         momentum=0.937,     # Momentum for SGD compatibility
#         weight_decay=0.0005,# L2 regularization
#         warmup_epochs=3,    # Gradual warmup
#         warmup_momentum=0.8,# Warmup momentum
#         cos_lr=True,        # NEW: Cosine annealing learning rate (smoother)

#         # ── Output & Validation ──────────────────────────────────
#         save=True,
#         save_period=10,     # Save checkpoint every 10 epochs
#         plots=True,         # Generate training plots
#         verbose=True,
#         val=True,           # Validate each epoch
#         patience=100,       # Early stopping patience (no improvement for 100 epochs)
#     )
#     return results


# # ─────────────────────────────────────────────────────────────────────
# # STEP 4 — Evaluate on Test Set
# # ─────────────────────────────────────────────────────────────────────

# def evaluate_model():
#     best_weights = f"{PROJECT_NAME}/{RUN_NAME}/weights/best.pt"
#     if not os.path.exists(best_weights):
#         print(f"\n[WARN] best.pt not found at {best_weights}. Skipping evaluation.")
#         return

#     print(f"\n[Step 4] Evaluating on test set...")
#     model   = YOLO(best_weights)
#     metrics = model.val(
#         data=str(AUG_YAML),
#         split="test",
#         imgsz=IMG_SIZE,
#         batch=BATCH_SIZE,
#         device=DEVICE,
#         project=PROJECT_NAME,
#         name=f"{RUN_NAME}_test_eval",
#         verbose=True,
#     )

#     print("\n  ── Test Set Results ──────────────────────────────")
#     print(f"  mAP@0.5      : {metrics.box.map50:.4f}")
#     print(f"  mAP@0.5:0.95 : {metrics.box.map:.4f}")
#     print(f"  Precision    : {metrics.box.mp:.4f}")
#     print(f"  Recall       : {metrics.box.mr:.4f}")
#     print("  ──────────────────────────────────────────────────")
#     return metrics


# # ─────────────────────────────────────────────────────────────────────
# # MAIN
# # ─────────────────────────────────────────────────────────────────────

# if __name__ == "__main__":
#     print("=" * 70)
#     print("  YOLOv8 OPTIMIZED Pole & Wire Detection — Colab Training Pipeline")
#     print("=" * 70)

#     # ── Verify all required folders exist ─────────────────────────────
#     required = {
#         "train/images" : TRAIN_IMG_DIR,
#         "train/labels" : TRAIN_LBL_DIR,
#         "valid/images" : VALID_IMG_DIR,
#         "test/images"  : TEST_IMG_DIR,
#     }
#     all_ok = True
#     print()
#     for name, path in required.items():
#         exists = path.exists()
#         print(f"  [{'OK' if exists else 'MISSING'}] {name}")
#         print(f"        → {path}")
#         if not exists:
#             all_ok = False

#     if not all_ok:
#         raise FileNotFoundError(
#             "\n[ERROR] One or more folders are missing.\n"
#             "Expected structure:\n"
#             "  /content/\n"
#             "  ├── train/images/\n"
#             "  ├── train/labels/\n"
#             "  ├── valid/images/\n"
#             "  ├── valid/labels/\n"
#             "  ├── test/images/\n"
#             "  ├── test/labels/\n"
#             "  └── data.yaml\n"
#         )

#     print()
#     fix_data_yaml()                     # Step 0 — Fix relative paths in data.yaml
#     apply_albumentations_to_dataset()   # Step 1 — Albumentations pre-processing
#     create_augmented_yaml()             # Step 2 — Create data_augmented.yaml
#     train_yolov8()                      # Step 3 — YOLOv8 OPTIMIZED training
#     evaluate_model()                    # Step 4 — Test set evaluation

#     print("\n  ═══════════════════════════════════════════════════════════════")
#     print("  All done! Your OPTIMIZED model is ready.")
#     print("  ═══════════════════════════════════════════════════════════════")
#     print(f"  Best weights : {PROJECT_NAME}/{RUN_NAME}/weights/best.pt")
#     print(f"  Last weights : {PROJECT_NAME}/{RUN_NAME}/weights/last.pt")
#     print(f"  Plots/results: {PROJECT_NAME}/{RUN_NAME}/")
#     print("\n  Expected improvements:")
#     print("  • mAP@0.5:0.95 should increase by +8–12 points")
#     print("  • Tighter bounding boxes (box=9.0 loss weight)")
#     print("  • Better pole edge detection (960px resolution)")
#     print("  • More stable training (cos_lr=True)")
#     print("\n  To use your model:")
#     print("    from ultralytics import YOLO")
#     print(f"    model = YOLO('{PROJECT_NAME}/{RUN_NAME}/weights/best.pt')")
#     print("    results = model.predict('image.jpg')")
#     print("  ═══════════════════════════════════════════════════════════════\n")

  YOLOv8 OPTIMIZED Pole & Wire Detection — Colab Training Pipeline

  [OK] train/images
        → /home/maska/NEW_poledata/train/images
  [OK] train/labels
        → /home/maska/NEW_poledata/train/labels
  [OK] valid/images
        → /home/maska/NEW_poledata/valid/images
  [OK] test/images
        → /home/maska/NEW_poledata/test/images

[Step 0] data.yaml fixed with absolute paths:
  train → /home/maska/NEW_poledata/train/images
  val   → /home/maska/NEW_poledata/valid/images
  test  → /home/maska/NEW_poledata/test/images

[Step 1] Found 8530 training images
         Generating 1 augmented copies each → 17060 total images



/home/maska/.conda/envs/torch_env/lib/python3.10/site-packages/albumentations/core/composition.py:331: UserWarning: Got processor for bboxes, but no transform to process it.
  self._set_keys()


  Processed 50/8530 images...
  Processed 100/8530 images...
  Processed 150/8530 images...
  Processed 200/8530 images...
  Processed 250/8530 images...
  Processed 300/8530 images...
  Processed 350/8530 images...
  Processed 400/8530 images...
  Processed 450/8530 images...
  Processed 500/8530 images...
  Processed 550/8530 images...
  Processed 600/8530 images...
  Processed 650/8530 images...
  Processed 700/8530 images...
  Processed 750/8530 images...
  Processed 800/8530 images...
  Processed 850/8530 images...
  Processed 900/8530 images...
  Processed 950/8530 images...
  Processed 1000/8530 images...
  Processed 1050/8530 images...
  Processed 1100/8530 images...
  Processed 1150/8530 images...
  Processed 1200/8530 images...
  Processed 1250/8530 images...
  Processed 1300/8530 images...
  Processed 1350/8530 images...
  Processed 1400/8530 images...
  Processed 1450/8530 images...
  Processed 1500/8530 images...
  Processed 1550/8530 images...
  Processed 1600/8530 images

/home/maska/.conda/envs/torch_env/lib/python3.10/site-packages/torch/utils/data/dataloader.py:617: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


optimizer: AdamW(lr=0.001, momentum=0.937) with parameter groups 97 weight(decay=0.0), 104 weight(decay=0.0005), 103 bias(decay=0.0)
Plotting labels to /nfs/storage1/home/maska/gsv_pole_detection1/exp1_optimized/labels.jpg... 
Image sizes 640 train, 640 val
Using 2 dataloader workers
Logging results to /nfs/storage1/home/maska/gsv_pole_detection1/exp1_optimized
Starting training for 100 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      1/100      10.8G      2.093      1.796      1.612         15        640: 100% ━━━━━━━━━━━━ 1600/1600 3.0it/s 8:58<0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 44/44 4.2it/s 10.6s0.2s
                   all       1399       1586      0.543      0.586      0.611      0.261

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      2/100      12.4G      2.007      1.715      1.572         11        640: 100% ━━━━━━━━━━━━

KeyboardInterrupt: 